# PPS surrogate statistical testing — production runner

Notebook này chạy tuần tự các session trong `SESSIONS`. Mỗi session chỉ log tiến độ, trạng thái checker và đường dẫn output; phần phân tích kết quả được thực hiện riêng.

In [2]:
# USER CONFIGURATION

RUN_PROCESSED = True
RUN_RAW = False
N_JOBS = 6
MASTER_SEED = 2026
ALLOW_OVERWRITE = False

# Imports and frozen runtime
from pathlib import Path
import sys
import time
from uuid import uuid4

import numpy as np
import pandas as pd


def find_project_root(start):
    """Find the repository root."""
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "phase1" / "src").is_dir():
            return path
    raise FileNotFoundError("Cannot locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
PHASE1_ROOT = PROJECT_ROOT / "phase1"
DATA_DIR = PHASE1_ROOT / "segmentated_data" / "dhdata"
RHO_DIR = (
    PHASE1_ROOT
    / "results"
    / "pps"
    / "pps_radius_calibration"
    / "csv"
)
OUTPUT_ROOT = PHASE1_ROOT / "results" / "statistic"
SESSION_ARCHIVE_TEMPLATE = "sample_{session_id}.npz"
SEGMENT_INDEX_FILENAME = "segments_index.csv"
RHO_FILENAME_TEMPLATE = "pps_radius_session_{session_id:02d}.csv"

WINDOW_SIZES = (60, 120, 180)
STATE_BATCHES = (("Awake", 0), ("Drowsy", 1))
EXPECTED_M = 39
IDENTITY_COLUMNS = [
    "session",
    "representation",
    "state",
    "window_size",
    "window_id",
]
REPRESENTATION_SETTINGS = {
    "Processed": ("processed", "processed"),
    "Raw": ("raw", "none"),
}

if str(PHASE1_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE1_ROOT))

from src.dataloader.loader import get_data
from src.surrogates.statistic_test import (
    METRIC_NAMES,
    SURROGATE_CONFIG,
    run_session_test,
)

assert SURROGATE_CONFIG["M"] == EXPECTED_M
assert N_JOBS == 6
assert MASTER_SEED == 2026


In [3]:
# INPUT PREPARATION
def _log(session_id, message):
    """Print one session-scoped progress message."""
    print(f"[Session {int(session_id):02d}] {message}", flush=True)


def _enabled_representations():
    """Return enabled representations in stable order."""
    enabled = []
    if RUN_PROCESSED:
        enabled.append("Processed")
    if RUN_RAW:
        enabled.append("Raw")
    if not enabled:
        raise ValueError("Enable at least one representation.")
    return tuple(enabled)


def _resolve_session_paths(session_id, representations):
    """Resolve required inputs and deterministic outputs."""
    archive = DATA_DIR / SESSION_ARCHIVE_TEMPLATE.format(
        session_id=session_id
    )
    index = DATA_DIR / SEGMENT_INDEX_FILENAME
    rho = RHO_DIR / RHO_FILENAME_TEMPLATE.format(
        session_id=session_id
    )
    required = {
        "segmented signal archive": archive,
        "window metadata index": index,
        "frozen rho lookup": rho,
    }
    missing = [
        f"{name}: {path}"
        for name, path in required.items()
        if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing required inputs:\n" + "\n".join(missing)
        )

    if representations == ("Processed",):
        output_dir = OUTPUT_ROOT / "processed"
        prefix = f"session_{session_id}_processed_pps"
    elif representations == ("Raw",):
        output_dir = OUTPUT_ROOT / "raw"
        prefix = f"session_{session_id}_raw_pps"
    else:
        output_dir = OUTPUT_ROOT
        prefix = f"session_{session_id}_pps"

    return {
        "session_stem": f"sample_{session_id}",
        "archive": archive,
        "index": index,
        "rho": rho,
        "output_dir": output_dir,
        "window_output": (
            output_dir / f"{prefix}_window_results.csv"
        ),
        "surrogate_output": (
            output_dir / f"{prefix}_surrogate_results.csv"
        ),
    }


def _append_window_records(
    records, session_id, representation, state, batches
):
    """Append one eligible representation-state cohort."""
    signal_key = REPRESENTATION_SETTINGS[representation][0]
    for window_size in WINDOW_SIZES:
        batch = batches[window_size]
        signals = batch[signal_key]
        window_ids = batch["window_id"]
        if len(signals) != len(window_ids):
            raise ValueError(
                "Signal and window identity counts are inconsistent."
            )
        for signal, window_id in zip(signals, window_ids, strict=True):
            records.append(
                {
                    "session": int(session_id),
                    "representation": representation,
                    "state": state,
                    "window_size": int(window_size),
                    "window_id": int(window_id),
                    "signal": np.asarray(signal, dtype=float),
                    "sampling_rate": float(batch["fs"]),
                }
            )


def _prepare_windows(session_id, representations, session_stem):
    """Load frozen eligible windows for one session."""
    records = []
    for representation in representations:
        _, stationarity = REPRESENTATION_SETTINGS[representation]
        state_data = get_data(
            session_stem,
            data_dir=DATA_DIR,
            window_sizes=WINDOW_SIZES,
            stationarity=stationarity,
        )
        for state, state_index in STATE_BATCHES:
            _append_window_records(
                records,
                session_id,
                representation,
                state,
                state_data[state_index],
            )
    windows = pd.DataFrame.from_records(records)
    if windows.empty:
        raise ValueError("No eligible windows were loaded.")
    return windows.sort_values(
        IDENTITY_COLUMNS, kind="stable"
    ).reset_index(drop=True)


def _load_rho_lookup(session_id, representations, rho_path):
    """Load and scope the frozen rho lookup."""
    required = [
        *IDENTITY_COLUMNS,
        "rho_star",
        "sampling_rate",
        "tau_samples",
        "boundary_flag",
    ]
    source = pd.read_csv(rho_path)
    missing = [column for column in required if column not in source]
    if missing:
        raise ValueError(f"Rho lookup is missing columns: {missing}")
    lookup = source.loc[
        source["session"].eq(session_id)
        & source["representation"].isin(representations)
    ].copy()
    if lookup.empty:
        raise ValueError("No rho rows match the enabled scope.")
    return lookup


In [4]:
# VALIDATION AND SAVE HELPERS
def _boundary_mask(values):
    """Parse boundary flags without implicit truth conversion."""
    normalized = values.astype(str).str.strip().str.casefold()
    valid = normalized.isin(("true", "false", "1", "0"))
    if not valid.all():
        invalid = sorted(normalized.loc[~valid].unique().tolist())
        raise ValueError(f"Invalid boundary_flag values: {invalid}")
    return normalized.isin(("true", "1"))


def _raise_failed_checks(stage, checks):
    """Log checks and raise when any checker fails."""
    for name, passed in checks.items():
        status = "PASS" if bool(passed) else "FAIL"
        print(f"    {name}: {status}", flush=True)
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise RuntimeError(
            f"{stage} failed: {', '.join(failed)}"
        )


def _validate_before_run(
    session_id, windows, rho_lookup, output_paths
):
    """Validate identities, rho coverage, and output safety."""
    coverage = windows[IDENTITY_COLUMNS].merge(
        rho_lookup[IDENTITY_COLUMNS],
        on=IDENTITY_COLUMNS,
        how="outer",
        indicator=True,
    )
    missing_rho = int(coverage["_merge"].eq("left_only").sum())
    extra_rho = int(coverage["_merge"].eq("right_only").sum())
    duplicate_count = int(
        windows.duplicated(IDENTITY_COLUMNS, keep=False).sum()
        + rho_lookup.duplicated(IDENTITY_COLUMNS, keep=False).sum()
    )
    boundary_count = int(
        _boundary_mask(rho_lookup["boundary_flag"]).sum()
    )
    rho_values = rho_lookup[
        ["rho_star", "sampling_rate", "tau_samples"]
    ].to_numpy(float)
    conflicts = [path for path in output_paths if path.exists()]
    checks = {
        "eligible windows available": len(windows) > 0,
        "window identities unique": duplicate_count == 0,
        "rho coverage complete": missing_rho == 0,
        "no extra rho identities": extra_rho == 0,
        "window and rho counts match": len(windows) == len(rho_lookup),
        "rho values finite and positive": bool(
            np.isfinite(rho_values).all() and (rho_values > 0).all()
        ),
        "no boundary cases": boundary_count == 0,
        "output paths available": (
            ALLOW_OVERWRITE or not conflicts
        ),
    }
    _log(session_id, "Pre-run checkers")
    _raise_failed_checks("Pre-run validation", checks)
    return {
        "missing_rho": missing_rho,
        "duplicates": duplicate_count,
        "boundary_cases": boundary_count,
    }


def _failure_count(results):
    """Count failure records attached by the statistical core."""
    failures = results.attrs.get("failures")
    if failures is None:
        return 0
    return len(failures)


def _validate_after_run(
    session_id, windows, window_results, surrogate_results
):
    """Validate complete window- and surrogate-level outputs."""
    expected_windows = len(windows)
    expected_surrogates = expected_windows * EXPECTED_M
    expected_ids = set(range(EXPECTED_M))
    groups = surrogate_results.groupby(
        IDENTITY_COLUMNS, observed=True
    ).agg(
        rows=("surrogate_id", "size"),
        unique_ids=("surrogate_id", "nunique"),
        unique_seeds=("seed", "nunique"),
    )
    id_sets = surrogate_results.groupby(
        IDENTITY_COLUMNS, observed=True
    )["surrogate_id"].agg(lambda values: set(map(int, values)))
    input_ids = windows[IDENTITY_COLUMNS].sort_values(
        IDENTITY_COLUMNS, kind="stable"
    ).reset_index(drop=True)
    result_ids = window_results[IDENTITY_COLUMNS].sort_values(
        IDENTITY_COLUMNS, kind="stable"
    ).reset_index(drop=True)
    surrogate_group_ids = (
        surrogate_results[IDENTITY_COLUMNS]
        .drop_duplicates()
        .sort_values(IDENTITY_COLUMNS, kind="stable")
        .reset_index(drop=True)
    )
    window_metric_columns = [
        f"{metric}_{suffix}"
        for metric in METRIC_NAMES
        for suffix in (
            "original",
            "surrogate_mean",
            "surrogate_sd",
            "surrogate_median",
            "rank",
            "p",
        )
    ]
    required_window_metrics = all(
        column in window_results for column in window_metric_columns
    )
    required_surrogate_metrics = all(
        metric in surrogate_results for metric in METRIC_NAMES
    )
    finite_metrics = bool(
        required_window_metrics
        and required_surrogate_metrics
        and np.isfinite(
            window_results[window_metric_columns].to_numpy(float)
        ).all()
        and np.isfinite(
            surrogate_results[list(METRIC_NAMES)].to_numpy(float)
        ).all()
    )
    failure_count = (
        _failure_count(window_results)
        + _failure_count(surrogate_results)
    )
    checks = {
        "window row count": len(window_results) == expected_windows,
        "window identities preserved": input_ids.equals(result_ids),
        "surrogate row count": (
            len(surrogate_results) == expected_surrogates
        ),
        "surrogate identities preserved": (
            input_ids.equals(surrogate_group_ids)
        ),
        "39 rows per window": bool(
            len(groups) == expected_windows
            and groups["rows"].eq(EXPECTED_M).all()
        ),
        "39 unique surrogate IDs per window": bool(
            groups["unique_ids"].eq(EXPECTED_M).all()
            and id_sets.map(lambda values: values == expected_ids).all()
        ),
        "39 unique seeds per window": bool(
            groups["unique_seeds"].eq(EXPECTED_M).all()
        ),
        "all metrics finite": finite_metrics,
        "no failure records": failure_count == 0,
    }
    _log(session_id, "Post-run checkers")
    _raise_failed_checks("Post-run integrity", checks)


def _save_results(
    session_id, paths, window_results, surrogate_results
):
    """Save both CSV files as one recoverable operation."""
    targets = (paths["window_output"], paths["surrogate_output"])
    conflicts = [path for path in targets if path.exists()]
    if conflicts and not ALLOW_OVERWRITE:
        formatted = "\n".join(str(path) for path in conflicts)
        raise FileExistsError(
            "Output exists. Delete it or set ALLOW_OVERWRITE=True:\n"
            f"{formatted}"
        )

    paths["output_dir"].mkdir(parents=True, exist_ok=True)
    token = uuid4().hex
    temporary = tuple(
        target.with_name(f".{target.name}.{token}.tmp")
        for target in targets
    )
    backups = tuple(
        target.with_name(f".{target.name}.{token}.bak")
        for target in targets
    )
    frames = (window_results, surrogate_results)
    installed = []
    backed_up = []
    try:
        for frame, temp_path in zip(frames, temporary, strict=True):
            frame.to_csv(temp_path, index=False)
            if len(pd.read_csv(temp_path)) != len(frame):
                raise RuntimeError(f"CSV verification failed: {temp_path}")
        for target, backup in zip(targets, backups, strict=True):
            if target.exists():
                target.replace(backup)
                backed_up.append((target, backup))
        for temp_path, target in zip(temporary, targets, strict=True):
            temp_path.replace(target)
            installed.append(target)
        for _, backup in backed_up:
            backup.unlink(missing_ok=True)
    except Exception:
        for target in installed:
            target.unlink(missing_ok=True)
        for target, backup in backed_up:
            if backup.exists():
                backup.replace(target)
        raise
    finally:
        for temp_path in temporary:
            temp_path.unlink(missing_ok=True)
        for backup in backups:
            backup.unlink(missing_ok=True)

    _log(session_id, "Save check: PASS")
    for target in targets:
        _log(session_id, f"Saved: {target}")


In [5]:
# SESSION RUNNER
def statistic_test_run(session_id):
    """Run, validate, and save one production session."""
    if isinstance(session_id, bool) or int(session_id) != session_id:
        raise TypeError("session_id must be an integer.")
    session_id = int(session_id)
    if session_id <= 0:
        raise ValueError("session_id must be positive.")

    started = time.perf_counter()
    representations = _enabled_representations()
    _log(
        session_id,
        f"START | representations={', '.join(representations)}",
    )
    try:
        paths = _resolve_session_paths(session_id, representations)
        _log(session_id, "Input files: PASS")

        windows = _prepare_windows(
            session_id, representations, paths["session_stem"]
        )
        rho_lookup = _load_rho_lookup(
            session_id, representations, paths["rho"]
        )
        _log(
            session_id,
            f"Inputs loaded | windows={len(windows)} "
            f"| rho_rows={len(rho_lookup)}",
        )
        _validate_before_run(
            session_id,
            windows,
            rho_lookup,
            (paths["window_output"], paths["surrogate_output"]),
        )

        _log(
            session_id,
            f"Core run: START | windows={len(windows)} "
            f"| PPS={len(windows) * EXPECTED_M} | n_jobs={N_JOBS}",
        )
        window_results, surrogate_results = run_session_test(
            windows_df=windows,
            rho_lookup_df=rho_lookup,
            session_id=session_id,
            master_seed=MASTER_SEED,
            n_jobs=N_JOBS,
            fail_fast=True,
        )
        _log(session_id, "Core run: PASS")

        _validate_after_run(
            session_id, windows, window_results, surrogate_results
        )
        _save_results(
            session_id, paths, window_results, surrogate_results
        )
    except Exception as exc:
        runtime = time.perf_counter() - started
        _log(
            session_id,
            f"STATUS: FAIL | runtime={runtime:.2f} s "
            f"| {type(exc).__name__}: {exc}",
        )
        raise

    runtime = time.perf_counter() - started
    _log(session_id, f"STATUS: PASS | runtime={runtime:.2f} s")
    return {
        "session_id": session_id,
        "window_output": paths["window_output"],
        "surrogate_output": paths["surrogate_output"],
    }


In [9]:
SESSIONS = [23]
print(len(SESSIONS))

1


In [10]:

# RUN SELECTED SESSIONS
for session_id in SESSIONS:
    statistic_test_run(session_id)


[Session 23] START | representations=Processed
[Session 23] Input files: PASS
[Session 23] Inputs loaded | windows=104 | rho_rows=104
[Session 23] Pre-run checkers
    eligible windows available: PASS
    window identities unique: PASS
    rho coverage complete: PASS
    no extra rho identities: PASS
    window and rho counts match: PASS
    rho values finite and positive: PASS
    no boundary cases: PASS
    output paths available: PASS
[Session 23] Core run: START | windows=104 | PPS=4056 | n_jobs=6
Session 23
Windows: 104
Workers: 6
M: 39
Completed: 1 / 104 | Elapsed: 55.9 s | Average/window: 55.9 s | Estimated remaining: 5753.2 s
Completed: 6 / 104 | Elapsed: 58.6 s | Average/window: 9.8 s | Estimated remaining: 957.8 s
Completed: 12 / 104 | Elapsed: 128.3 s | Average/window: 10.7 s | Estimated remaining: 983.9 s
Completed: 18 / 104 | Elapsed: 198.6 s | Average/window: 11.0 s | Estimated remaining: 949.1 s
Completed: 24 / 104 | Elapsed: 267.9 s | Average/window: 11.2 s | Estimated 